In [ ]:
# CELL 1 - SETUP: copy the code from your private Kaggle code dataset, install it, show the GPU
import glob, importlib, os, re, shutil, subprocess, sys

# Two private datasets are needed: the CODE (the unzipped genai_code.zip) and the Task 2 SOURCES (the unzipped t3_sources.zip).
# Kaggle may mount them under nested folders, so look for the files anywhere under /kaggle/input.
found = glob.glob("/kaggle/input/**/tools/t3_pipeline.py", recursive=True)
assert found, "code dataset not found (or an old one without tools/t3_pipeline.py): Add Input / upload genai_code.zip as a NEW dataset"
sources = glob.glob("/kaggle/input/**/t2_classifier.pt", recursive=True)
assert sources, "sources dataset not found: Add Input > the dataset made from t3_sources.zip (it holds t2_classifier.pt)"
CODE_DIR = os.path.dirname(os.path.dirname(found[0]))
print("code found in", CODE_DIR)
print("code version:", re.search(r'PIPELINE_VERSION = "([^"]+)"', open(found[0]).read()).group(1))   # the docs say which value is expected
print("sources found:", sources)

# always refresh the code; dirs_exist_ok keeps the artifacts/ folder of an earlier run, so a resume finds its state
shutil.copytree(CODE_DIR, "/kaggle/working/repo", dirs_exist_ok=True)
os.chdir("/kaggle/working/repo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "-r", "requirements_kaggle.txt"])
importlib.invalidate_caches()
subprocess.call(["nvidia-smi"])


In [ ]:
# CELL 2 - PARAMETERS: the only cell you normally edit
DEVICE_PROFILE = "kaggle"     # picks configs/devices/kaggle.yaml (data under /kaggle/input, outputs in /kaggle/working)
TAG = "t3"                    # names the log/state folder artifacts/logs/<TAG>/ (keep it when you re-run to RESUME)
DRY_RUN = True                # True = tiny end-to-end check (a few minutes); then set False for the real run
WANDB_SECRET = "WANDB_API_KEY"   # name of the Kaggle secret that holds your Weights & Biases API key
# Budget overrides, only if the `prepare` stage says the projected time is above the limit. One "KEY=VALUE" per entry,
# cut in this order (docs/TASK3_PLAN.md section C): "timeout_minutes=10", "train.joint_epochs=8", "epochs_per_trial.joint=2", "train.val_every_epochs=2"
BUDGET_OVERRIDES = []


In [ ]:
# CELL 3 - RUN: the supervised Task 3 pipeline
#   prepare (sources check + benchmark + time gate) -> Optuna study -> final training
#   -> val evaluation (T1, T2, T3 on the same tensors) -> ONNX export + parity + promote
# No logic lives in this notebook; tools/t3_pipeline.py restarts crashed stages and resumes where it stopped.
import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(WANDB_SECRET)
except Exception as err:
    print("No W&B API key found in the Kaggle secrets (", err, ") -> the runner will log offline.")

cmd = [sys.executable, "tools/t3_pipeline.py", "--device-profile", DEVICE_PROFILE, "--tag", TAG] + (["--dry-run"] if DRY_RUN else [])
for item in BUDGET_OVERRIDES:
    cmd += ["--set", item]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
print("runner exit code:", proc.wait())


In [ ]:
# CELL 4 - OUTPUTS: bundle everything you need into ONE file, then download it (Output tab after Save Version, or the file panel)
import shutil
from pathlib import Path

root, stage = Path("/kaggle/working/repo"), Path("/kaggle/working/t3_results")
shutil.rmtree(stage, ignore_errors=True)
stage.mkdir(parents=True)
items = ["studies/t3_moe", "configs/task3_moe_final.yaml",
         "models", "report/tables/onnx_parity.csv", f"artifacts/logs/{TAG}", "artifacts/optuna/t3_moe.db",
         "/kaggle/working/runs/task3", "/kaggle/working/eval/task3"]       # final run + evaluation (trial runs are left out)
for item in items:
    src = Path(item) if item.startswith("/") else root / item
    if src.exists():
        dst = stage / (src.relative_to("/kaggle/working") if item.startswith("/") else src.relative_to(root))
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
dry = root / "artifacts/dryrun_t3"                                            # exists only after a dry run (checkpoints and ONNX left out)
if dry.exists():
    shutil.copytree(dry, stage / "artifacts/dryrun_t3", ignore=shutil.ignore_patterns("*.pt", "*.onnx"))
shutil.make_archive("/kaggle/working/t3_results", "zip", stage)
print(f"{Path('/kaggle/working/t3_results.zip').stat().st_size / 2**20:.1f} MB: /kaggle/working/t3_results.zip")
log_dir = root / ("artifacts/dryrun_t3/logs" if DRY_RUN else f"artifacts/logs/{TAG}")
print((log_dir / "t3_status.log").read_text()[-3000:])
